In [1]:
d = "dataset_message_1to1.csv"

In [7]:
import pandas as pd


def print_label_counts_by_hashtag(csv_path, hashtag_col="hashtag", label_col="label"):
    df = pd.read_csv(csv_path)

    if hashtag_col not in df.columns:
        raise ValueError(f"Column '{hashtag_col}' not found in {csv_path}")
    if label_col not in df.columns:
        raise ValueError(f"Column '{label_col}' not found in {csv_path}")

    for hashtag, group in df.groupby(hashtag_col):
        num_ones = (group[label_col] == 1).sum()
        num_zeroes = (group[label_col] == 0).sum()
        ratio = num_ones / len(group) if len(group) > 0 else 0.0

        print(f"Hashtag: {hashtag}")
        print(f"  Ones: {num_ones}")
        print(f"  Zeroes: {num_zeroes}")
        print(f"  Total: {len(group)}")
        print(f"  Ratio of ones: {ratio:.4f}")
        print()


print_label_counts_by_hashtag(f"../data/processed/datasets/{d}")

Hashtag: AI
  Ones: 1261
  Zeroes: 1261
  Total: 2522
  Ratio of ones: 0.5000

Hashtag: Anime
  Ones: 2127
  Zeroes: 2127
  Total: 4254
  Ratio of ones: 0.5000

Hashtag: BlackHistoryMonth
  Ones: 3587
  Zeroes: 3587
  Total: 7174
  Ratio of ones: 0.5000

Hashtag: Booksky
  Ones: 2752
  Zeroes: 2752
  Total: 5504
  Ratio of ones: 0.5000

Hashtag: Gaza
  Ones: 2178
  Zeroes: 2178
  Total: 4356
  Ratio of ones: 0.5000

Hashtag: ICE
  Ones: 2472
  Zeroes: 2472
  Total: 4944
  Ratio of ones: 0.5000

Hashtag: Pokemon
  Ones: 3854
  Zeroes: 3854
  Total: 7708
  Ratio of ones: 0.5000

Hashtag: Superbowl
  Ones: 1726
  Zeroes: 1726
  Total: 3452
  Ratio of ones: 0.5000

Hashtag: TheTraitors
  Ones: 1071
  Zeroes: 1071
  Total: 2142
  Ratio of ones: 0.5000

Hashtag: Trump
  Ones: 2093
  Zeroes: 2093
  Total: 4186
  Ratio of ones: 0.5000



In [6]:


def enforce_exact_label_ratio_from_csv(
    neg_per_pos: int,
    csv_path: str = f"../data/processed/datasets/{d}",
    hashtag_col: str = "hashtag",
    label_col: str = "label",
    save_path: str = f"../data/processed/datasets/{d}",
    random_state: int = 42,
) -> pd.DataFrame:
    """
    For each hashtag, enforce the exact ratio:
        positives : negatives = 1 : neg_per_pos

    This is done by randomly removing either negatives or positives.

    Example:
        neg_per_pos=1 -> 1:1
        neg_per_pos=5 -> 1:5
    """
    df = pd.read_csv(csv_path)

    kept_parts = []

    for _, group in df.groupby(hashtag_col, sort=False):
        positives = group[group[label_col] == 1]
        negatives = group[group[label_col] == 0]

        n_pos = len(positives)
        n_neg = len(negatives)

        if n_pos == 0 or n_neg == 0:
            continue

        # Largest feasible exact ratio subset
        keep_pos = min(n_pos, n_neg // neg_per_pos)
        keep_neg = keep_pos * neg_per_pos

        if keep_pos == 0 or keep_neg == 0:
            continue

        positives = positives.sample(n=keep_pos, random_state=random_state)
        negatives = negatives.sample(n=keep_neg, random_state=random_state)

        balanced_group = pd.concat([positives, negatives]).sort_index()
        kept_parts.append(balanced_group)

    if not kept_parts:
        result = pd.DataFrame(columns=df.columns)
    else:
        result = pd.concat(kept_parts).sort_index().reset_index(drop=True)

    result.to_csv(save_path, index=False)
    return result
enforce_exact_label_ratio_from_csv(neg_per_pos=1)

,A_id,S_id,P_id,hashtag,label,M-text_len,M-word_count,M-neg,M-neu,M-pos,...,M-travel_&_adventure,M-youth_&_student_life,M-single_topic_count,M-single_topic_overall,M-single_arts_&_culture,M-single_business_&_entrepreneurs,M-single_pop_culture,M-single_daily_life,M-single_sports_&_gaming,M-single_science_&_technology
0,did:plc:rgn3wjhjtfbtaf55ewqbjb4j,did:plc:pwo2tzlabqaxzbwrx65pzsk3,at://did:plc:pwo2tzlabqaxzbwrx65pzsk3/app.bsky...,Trump,1,189,23,0.118,0.772,0.110,...,0.001883,0.001260,3,3,0.235138,0.354552,0.755440,0.803142,0.242252,0.578998
1,did:plc:rzk2b2piewxqujsm43f63lkm,did:plc:lopecfty2m5hwnlqls2ijfkh,at://did:plc:lopecfty2m5hwnlqls2ijfkh/app.bsky...,Trump,0,49,8,0.516,0.484,0.000,...,0.002356,0.001154,3,4,0.269242,0.175002,0.770728,0.723566,0.916042,0.118422
2,did:plc:gdg3slu366jwtvopjiujpplr,did:plc:tqma3outiazw5tx4vo7yjb4d,at://did:plc:tqma3outiazw5tx4vo7yjb4d/app.bsky...,Trump,1,261,50,0.279,0.647,0.074,...,0.001875,0.001425,3,3,0.221609,0.354913,0.757450,0.839478,0.253458,0.596711
3,did:plc:s5wiapmuhdqppcacqb4tynre,did:plc:hvaqzikyzu4e3vhlxdj7evj4,at://did:plc:hvaqzikyzu4e3vhlxdj7evj4/app.bsky...,Trump,1,157,27,0.130,0.870,0.000,...,0.001714,0.001217,2,2,0.283946,0.250483,0.878036,0.787257,0.349604,0.391556
4,did:plc:7wdc4zowcyzk6nuoqkvyhyu4,did:plc:lxnd5tcexalgikqzhehjwpjn,at://did:plc:lxnd5tcexalgikqzhehjwpjn/app.bsky...,Trump,0,278,57,0.192,0.751,0.056,...,0.001407,0.001167,3,2,0.194060,0.301135,0.762233,0.751854,0.257594,0.755154
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
46237,did:plc:pt4gk7ycrpe6wq6mz5ivuct5,did:plc:blufufhw6ak7hoew2yzyz2uw,at://did:plc:blufufhw6ak7hoew2yzyz2uw/app.bsky...,AI,0,260,36,0.000,0.831,0.169,...,0.037536,0.079257,3,5,0.622131,0.385853,0.589727,0.472813,0.203276,0.877967
46238,did:plc:3mvwwv4q3aehb46yk7zgrzsh,did:plc:jwcww7dqsvu4pklygynciiph,at://did:plc:jwcww7dqsvu4pklygynciiph/app.bsky...,AI,1,226,42,0.000,0.918,0.082,...,0.017029,0.010842,2,5,0.159162,0.387047,0.382036,0.372177,0.826744,0.900202
46239,did:plc:u3dommifjex6iy3pnp7onlq2,did:plc:ztvruzys53zgfjbsqchf34rp,at://did:plc:ztvruzys53zgfjbsqchf34rp/app.bsky...,AI,0,121,14,0.000,0.500,0.500,...,0.015587,0.007729,4,2,0.692191,0.373518,0.749201,0.524430,0.165674,0.674929
46240,did:plc:3aouxuhlujxrqjrxspkzhth5,did:plc:o3vijl7j6swjvf6tovdlqwpm,at://did:plc:o3vijl7j6swjvf6tovdlqwpm/app.bsky...,AI,1,248,42,0.000,0.897,0.103,...,0.037894,0.457084,2,5,0.277905,0.589398,0.322010,0.424701,0.247041,0.958591
